# 使用PyTorch实现LSTM进行文本分类

在这个Notebook中，我们将使用PyTorch实现一个LSTM模型来进行文本分类。我们将使用IMDB电影评论数据集，这是一个二分类任务，目标是预测电影评论的情感（正面或负面）。

## 1. 导入所需的库

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

# 移除torchtext相关的导入，改用其他库
from collections import Counter
import re
import random
import time
import os
import requests
from io import BytesIO
from zipfile import ZipFile

# 设置随机种子，确保结果可复现
SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
random.seed(SEED)
np.random.seed(SEED)

# 检查GPU是否可用
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

## 2. 加载和预处理IMDB数据集

In [ ]:
# 定义数据加载函数
def download_and_extract_imdb():
    """下载并解压IMDB数据集到data文件夹"""
    # IMDB数据集URL
    url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
    
    # 创建data目录（如果不存在）
    data_dir = os.path.join(os.getcwd(), "../data")
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)
    
    # 在data目录下创建aclImdb文件夹
    imdb_dir = os.path.join(data_dir, "aclImdb")
    
    # 检查数据集是否已经存在
    if not os.path.exists(imdb_dir):
        print(f"下载IMDB数据集到 {data_dir}...")
        # 创建临时文件来保存下载的数据
        import tarfile
        import tempfile
        
        # 下载数据
        response = requests.get(url, stream=True)
        
        # 创建临时文件
        with tempfile.NamedTemporaryFile(delete=False) as tmp_file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    tmp_file.write(chunk)
            tmp_file_path = tmp_file.name
        
        # 解压数据到data目录
        with tarfile.open(tmp_file_path) as tar:
            tar.extractall(path=data_dir)
        
        # 删除临时文件
        os.remove(tmp_file_path)
        print(f"IMDB数据集已下载并解压到 {imdb_dir}")
    else:
        print(f"IMDB数据集已存在于 {imdb_dir}")
    
    return imdb_dir

def load_imdb_data(data_dir):
    """从data文件夹加载IMDB数据集"""
    reviews = []
    labels = []
    
    # 加载训练集正面评论
    pos_files = os.path.join(data_dir, "train", "pos")
    for filename in os.listdir(pos_files):
        if filename.endswith(".txt"):
            with open(os.path.join(pos_files, filename), "r", encoding="utf-8") as f:
                reviews.append(f.read())
                labels.append(1)  # 正面评论标记为1
    
    # 加载训练集负面评论
    neg_files = os.path.join(data_dir, "train", "neg")
    for filename in os.listdir(neg_files):
        if filename.endswith(".txt"):
            with open(os.path.join(neg_files, filename), "r", encoding="utf-8") as f:
                reviews.append(f.read())
                labels.append(0)  # 负面评论标记为0
    
    # 创建训练数据DataFrame
    train_df = pd.DataFrame({
        "review": reviews,
        "sentiment": labels
    })
    
    # 清空列表以加载测试集
    reviews = []
    labels = []
    
    # 加载测试集正面评论
    pos_files = os.path.join(data_dir, "test", "pos")
    for filename in os.listdir(pos_files):
        if filename.endswith(".txt"):
            with open(os.path.join(pos_files, filename), "r", encoding="utf-8") as f:
                reviews.append(f.read())
                labels.append(1)  # 正面评论标记为1
    
    # 加载测试集负面评论
    neg_files = os.path.join(data_dir, "test", "neg")
    for filename in os.listdir(neg_files):
        if filename.endswith(".txt"):
            with open(os.path.join(neg_files, filename), "r", encoding="utf-8") as f:
                reviews.append(f.read())
                labels.append(0)  # 负面评论标记为0
    
    # 创建测试数据DataFrame
    test_df = pd.DataFrame({
        "review": reviews,
        "sentiment": labels
    })
    
    print(f"训练集大小: {len(train_df)}")
    print(f"测试集大小: {len(test_df)}")
    
    return train_df, test_df

# 创建数据预处理函数
def preprocess_text(text):
    """清洗文本数据"""
    # 转为小写
    text = text.lower()
    # 移除HTML标签
    text = re.sub('<.*?>', '', text)
    # 只保留字母和空格
    text = re.sub('[^a-z\s]', '', text)
    # 移除多余空格
    text = re.sub('\s+', ' ', text).strip()
    return text

# 定义分词函数
def tokenize(text):
    """简单的分词函数"""
    return text.split()

# 下载并加载数据集
try:
    data_dir = download_and_extract_imdb()
    train_df, test_df = load_imdb_data(data_dir)
    
    # 预处理文本
    train_df['processed_review'] = train_df['review'].apply(preprocess_text)
    test_df['processed_review'] = test_df['review'].apply(preprocess_text)
    
    # 拆分训练集为训练集和验证集
    train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=SEED)
    
    print(f"训练集大小: {len(train_df)}")
    print(f"验证集大小: {len(val_df)}")
    print(f"测试集大小: {len(test_df)}")
except Exception as e:
    print(f"加载数据时出错: {e}")
    # 创建一些示例数据以便代码可以继续运行
    print("创建示例数据以演示模型...")
    
    # 创建简单的示例数据集
    example_reviews = [
        "This movie was great! I loved it.",
        "Amazing film with wonderful actors.",
        "I really enjoyed watching this movie.",
        "This was a terrible waste of time.",
        "I hated this movie so much.",
        "The worst film I've ever seen."
    ]
    example_sentiments = [1, 1, 1, 0, 0, 0]  # 1=正面, 0=负面
    
    # 创建扩展版本的数据集
    extended_reviews = example_reviews * 50  # 重复以创建更多样本
    extended_sentiments = example_sentiments * 50
    
    # 创建DataFrame
    full_df = pd.DataFrame({
        "review": extended_reviews,
        "sentiment": extended_sentiments,
        "processed_review": [preprocess_text(text) for text in extended_reviews]
    })
    
    # 拆分为训练、验证和测试集
    train_val_df, test_df = train_test_split(full_df, test_size=0.2, random_state=SEED)
    train_df, val_df = train_test_split(train_val_df, test_size=0.2, random_state=SEED)
    
    print(f"训练集大小: {len(train_df)}")
    print(f"验证集大小: {len(val_df)}")
    print(f"测试集大小: {len(test_df)}")

## 3. 构建词汇表和数据加载器

In [ ]:
# 自定义词汇表构建函数
class Vocab:
    def __init__(self, min_freq=5):
        self.min_freq = min_freq
        self.itos = {0: "<pad>", 1: "<unk>"}  # index to string
        self.stoi = {"<pad>": 0, "<unk>": 1}  # string to index
    
    def build_vocab(self, texts):
        """从文本列表构建词汇表"""
        counter = Counter()
        for text in texts:
            counter.update(tokenize(text))
        
        # 仅保留出现次数达到最小频率的词
        words = [word for word, count in counter.items() if count >= self.min_freq]
        
        # 添加到词汇表
        for i, word in enumerate(words, start=len(self.itos)):
            self.itos[i] = word
            self.stoi[word] = i
        
        return self
    
    def numericalize(self, text):
        """将文本转换为数字序列"""
        tokenized = tokenize(text)
        return [self.stoi.get(token, self.stoi["<unk>"]) for token in tokenized]
    
    def __len__(self):
        return len(self.itos)

# 创建自定义数据集
class IMDBDataset(Dataset):
    def __init__(self, dataframe, vocab, max_length=256):
        self.reviews = dataframe['processed_review'].values
        self.labels = dataframe['sentiment'].values
        self.vocab = vocab
        self.max_length = max_length
        
    def __len__(self):
        return len(self.reviews)
    
    def __getitem__(self, idx):
        text = self.reviews[idx]
        label = self.labels[idx]
        
        # 将文本转换为数字序列
        indexed = self.vocab.numericalize(text)
        
        # 截断或填充到固定长度
        if len(indexed) > self.max_length:
            indexed = indexed[:self.max_length]
        else:
            indexed = indexed + [0] * (self.max_length - len(indexed))
            
        return torch.tensor(indexed), label

# 构建词汇表
vocab = Vocab(min_freq=5)
vocab.build_vocab(train_df['processed_review'].values)
print(f"词汇表大小: {len(vocab)}")

# 创建数据加载器
BATCH_SIZE = 64
MAX_LENGTH = 256

train_dataset = IMDBDataset(train_df, vocab, MAX_LENGTH)
val_dataset = IMDBDataset(val_df, vocab, MAX_LENGTH)
test_dataset = IMDBDataset(test_df, vocab, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

## 4. 实现LSTM模型

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout, pad_idx):
        super().__init__()
        
        # 嵌入层
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        
        # LSTM层
        self.lstm = nn.LSTM(embedding_dim, 
                           hidden_dim, 
                           num_layers=n_layers, 
                           bidirectional=bidirectional, 
                           dropout=dropout,
                           batch_first=True)
        
        # 确定全连接层的输入大小
        fc_in_dim = hidden_dim * 2 if bidirectional else hidden_dim
        
        # 全连接层
        self.fc = nn.Linear(fc_in_dim, output_dim)
        
        # Dropout层
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, text):
        # text: [batch_size, seq_len]
        
        # 对文本进行嵌入
        embedded = self.dropout(self.embedding(text))  # [batch_size, seq_len, emb_dim]
        
        # 通过LSTM层
        output, (hidden, cell) = self.lstm(embedded)  # output: [batch_size, seq_len, hid_dim * num_directions]
                                                     # hidden: [num_layers * num_directions, batch_size, hid_dim]
        
        # 对于分类任务，我们通常只需要最后一个时间步的隐藏状态
        if self.lstm.bidirectional:
            # 如果是双向LSTM，拼接最后一层的前向和后向隐藏状态
            hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)  # [batch_size, hid_dim * 2]
        else:
            hidden = hidden[-1]  # [batch_size, hid_dim]
            
        # 应用dropout并通过全连接层
        hidden = self.dropout(hidden)
        output = self.fc(hidden)  # [batch_size, output_dim]
        
        return output

# 设置模型参数
INPUT_DIM = len(vocab)  # 词汇表大小
EMBEDDING_DIM = 100    # 嵌入层维度
HIDDEN_DIM = 256       # 隐藏层维度
OUTPUT_DIM = 1         # 输出维度 (二分类)
N_LAYERS = 2           # LSTM层数
BIDIRECTIONAL = True   # 是否使用双向LSTM
DROPOUT = 0.5          # Dropout率
PAD_IDX = vocab.stoi['<pad>']  # padding索引

# 实例化模型
model = LSTMClassifier(
    INPUT_DIM, 
    EMBEDDING_DIM, 
    HIDDEN_DIM, 
    OUTPUT_DIM, 
    N_LAYERS, 
    BIDIRECTIONAL, 
    DROPOUT, 
    PAD_IDX
)

# 定义损失函数和优化器
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 将模型移至GPU（如果可用）
model = model.to(device)
criterion = criterion.to(device)

# 打印模型结构
print(model)

## 5. 训练函数

In [ ]:
def binary_accuracy(preds, y):
    """
    计算二分类精度
    """
    # 对预测结果进行四舍五入，转为0或1
    rounded_preds = torch.round(torch.sigmoid(preds))
    correct = (rounded_preds == y).float()
    acc = correct.sum() / len(correct)
    return acc

def train(model, dataloader, optimizer, criterion, device):
    """
    训练一个epoch
    """
    epoch_loss = 0
    epoch_acc = 0
    
    model.train()
    
    for texts, labels in dataloader:
        # 将数据移至设备
        texts = texts.to(device)
        labels = labels.float().to(device)
        
        # 清除梯度
        optimizer.zero_grad()
        
        # 前向传播
        predictions = model(texts).squeeze(1)
        
        # 计算损失
        loss = criterion(predictions, labels)
        
        # 计算精度
        acc = binary_accuracy(predictions, labels)
        
        # 反向传播
        loss.backward()
        
        # 更新参数
        optimizer.step()
        
        epoch_loss += loss.item()
        epoch_acc += acc.item()
        
    return epoch_loss / len(dataloader), epoch_acc / len(dataloader

def evaluate(model, dataloader, criterion, device):
    """
    评估模型
    """
    epoch_loss = 0
    epoch_acc = 0
    
    model.eval()
    
    with torch.no_grad():
        for texts, labels in dataloader:
            # 将数据移至设备
            texts = texts.to(device)
            labels = labels.float().to(device)
            
            # 前向传播
            predictions = model(texts).squeeze(1)
            
            # 计算损失
            loss = criterion(predictions, labels)
            
            # 计算精度
            acc = binary_accuracy(predictions, labels)
            
            epoch_loss += loss.item()
            epoch_acc += acc.item()
        
    return epoch_loss / len(dataloader), epoch_acc / len(dataloader

## 6. 训练模型

In [ ]:
# 定义训练参数
N_EPOCHS = 5
best_valid_loss = float('inf')

# 训练循环
for epoch in range(N_EPOCHS):
    
    start_time = time.time()
    
    # 训练和评估
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
    valid_loss, valid_acc = evaluate(model, val_loader, criterion, device)
    
    end_time = time.time()
    
    epoch_mins, epoch_secs = divmod(end_time - start_time, 60)
    
    # 如果验证损失更好，保存模型
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'lstm-model.pt')
    
    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs:.2f}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

## 7. 评估模型

In [ ]:
# 加载最佳模型
model.load_state_dict(torch.load('lstm-model.pt'))

# 在测试集上评估
test_loss, test_acc = evaluate(model, test_loader, criterion, device)

print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')

# 详细评估：获取预测结果和真实标签
y_pred = []
y_true = []

model.eval()
with torch.no_grad():
    for texts, labels in test_loader:
        texts = texts.to(device)
        predictions = model(texts).squeeze(1)
        predictions = torch.sigmoid(predictions)
        predictions = torch.round(predictions)
        
        y_pred.extend(predictions.cpu().numpy())
        y_true.extend(labels.numpy())

# 生成分类报告
print("分类报告:")
print(classification_report(y_true, y_pred, target_names=['负面', '正面']))

# 绘制混淆矩阵
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['负面', '正面'], yticklabels=['负面', '正面'])
plt.xlabel('预测标签')
plt.ylabel('真实标签')
plt.title('混淆矩阵')
plt.show()

## 8. 使用模型进行预测

In [ ]:
def predict_sentiment(model, text, vocab, device, max_length=256):
    """预测单个文本的情感"""
    model.eval()
    
    # 预处理文本
    text = preprocess_text(text)
    # 将文本转换为数字序列
    indexed = vocab.numericalize(text)
    
    # 截断或填充
    if len(indexed) > max_length:
        indexed = indexed[:max_length]
    else:
        indexed = indexed + [0] * (max_length - len(indexed))
    
    # 转为张量并移至设备
    tensor = torch.LongTensor(indexed).unsqueeze(0).to(device)  # [1, max_length]
    
    # 预测
    with torch.no_grad():
        prediction = torch.sigmoid(model(tensor).squeeze(1))
    
    # 返回概率和预测类别
    probability = prediction.item()
    sentiment = "正面" if probability >= 0.5 else "负面"
    
    return sentiment, probability

# 示例预测
test_examples = [
    "This movie was amazing! I loved every minute of it.",
    "The acting was terrible and the plot made no sense.",
    "This film is neither good nor bad, just mediocre.",
    "The special effects were good but the storyline was weak."
]

for example in test_examples:
    sentiment, probability = predict_sentiment(model, example, vocab, device)
    print(f"文本: {example}")
    print(f"情感: {sentiment} (概率: {probability:.4f})")
    print("-" * 80)

## 9. 总结与改进方向

在这个notebook中，我们实现了一个基于LSTM的文本分类模型，使用PyTorch框架进行构建和训练。我们使用了IMDB电影评论数据集，构建了一个能够判断评论情感的二分类模型。

### 可能的改进方向：

1. **预训练词嵌入**：使用GloVe或Word2Vec等预训练词嵌入可以提高模型性能。
2. **注意力机制**：在LSTM层上添加注意力机制，关注更重要的单词或短语。
3. **更复杂的架构**：尝试其他架构，如GRU、双向LSTM+CNN等。
4. **正则化技术**：增加正则化方法如权重衰减、提前停止等来减少过拟合。
5. **超参数调优**：使用网格搜索或贝叶斯优化调整超参数。
6. **数据增强**：通过同义词替换、回译等方法增强训练数据。
7. **更现代的方法**：尝试使用Transformer架构，如BERT、RoBERTa等预训练模型进行微调。

循环神经网络，尤其是LSTM，在处理序列数据（如文本）时非常有效，因为它们能够捕获长距离依赖关系。但现代NLP已经开始转向基于Transformer的模型，它们在许多任务上表现更好。不过，LSTM仍然是一个重要的基础模型，了解其工作原理对深入学习深度学习非常有价值。